# ДЗ 1

In [ ]:
!nvidia-smi
!pip -q install nvidia-ml-py

In [ ]:
import csv
import gc
import glob
import json
import math
import os
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Image, display
from matplotlib.colors import LogNorm
from scipy.optimize import least_squares, nnls
from torch.utils.flop_counter import FlopCounterMode

## Модель

In [ ]:
def conv_relu(c_in, c_out, k, stride=1):
    return [
        nn.Conv2d(c_in, c_out, k, stride=stride, padding=k // 2, bias=False),
        nn.ReLU(inplace=True),
    ]


def make_model(num_classes=100):
    return nn.Sequential(
        *conv_relu(3, 32, 7, stride=2),
        nn.MaxPool2d(3, stride=2, padding=1),
        *conv_relu(32, 64, 5),
        *conv_relu(64, 128, 3, stride=2),
        *conv_relu(128, 256, 1),
        *conv_relu(256, 256, 3, stride=2),
        *conv_relu(256, 512, 1),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(512, 256),
        nn.ReLU(inplace=True),
        nn.Linear(256, num_classes),
    )

## Формулы

In [ ]:
# имя, тип, FLOPs на B*S^2, FLOPs на картинку, байты на B*S^2, байты на картинку, байты весов
LAYERS = [
    ("conv1", "conv", 2352, 0, 44, 0, 18816),
    ("relu1", "mem", 0, 0, 64, 0, 0),
    ("pool", "mem", 0, 0, 40, 0, 0),
    ("conv2", "conv", 6400, 0, 24, 0, 204800),
    ("relu2", "mem", 0, 0, 32, 0, 0),
    ("conv3", "conv", 2304, 0, 24, 0, 294912),
    ("relu3", "mem", 0, 0, 16, 0, 0),
    ("conv4", "conv", 1024, 0, 24, 0, 131072),
    ("relu4", "mem", 0, 0, 32, 0, 0),
    ("conv5", "conv", 4608, 0, 20, 0, 2359296),
    ("relu5", "mem", 0, 0, 8, 0, 0),
    ("conv6", "conv", 1024, 0, 12, 0, 524288),
    ("relu6", "mem", 0, 0, 16, 0, 0),
    ("gap", "mem", 0, 0, 8, 2048, 0),
    ("fc1", "gemm", 0, 262144, 0, 3072, 525312),
    ("relu7", "mem", 0, 0, 0, 2048, 0),
    ("fc2", "gemm", 0, 51200, 0, 1424, 102800),
]
N = len(LAYERS)
F_X = np.array([l[2] for l in LAYERS], dtype=float)
F_B = np.array([l[3] for l in LAYERS], dtype=float)
M_X = np.array([l[4] for l in LAYERS], dtype=float)
M_B = np.array([l[5] for l in LAYERS], dtype=float)
W = np.array([l[6] for l in LAYERS], dtype=float)

WEIGHTS = 4161296


def _arr(image_size, batch):
    return np.asarray(image_size, dtype=float), np.asarray(batch, dtype=float)


def flops(image_size, batch):
    s, b = _arr(image_size, batch)
    return b * (17712 * s**2 + 313344)


def bytes_moved(image_size, batch):
    s, b = _arr(image_size, batch)
    return 364 * b * s**2 + 8592 * b + WEIGHTS


def memory(image_size, batch):
    s, b = _arr(image_size, batch)
    return WEIGHTS + 52 * b * s**2


def max_batch(image_size, budget):
    s = np.asarray(image_size, dtype=float)
    return np.floor((budget - WEIGHTS) / (52 * s**2))


def layer_costs(image_size, batch):
    s, b = _arr(image_size, batch)
    x, b = np.broadcast_arrays(b * s**2, b)
    shape = (N,) + (1,) * x.ndim
    f = F_X.reshape(shape) * x + F_B.reshape(shape) * b
    m = M_X.reshape(shape) * x + M_B.reshape(shape) * b + W.reshape(shape)
    return f, m


def layer_times(image_size, batch, theta):
    f, m = layer_costs(image_size, batch)
    return np.maximum(m / theta["bandwidth"], f / theta["throughput"])


def latency(image_size, batch, theta):
    g = layer_times(image_size, batch, theta)
    rest = np.cumsum(g[::-1], axis=0)[::-1]
    k = np.arange(1, N + 1).reshape((N,) + (1,) * (g.ndim - 1))
    return theta["t0"] + np.max(k * theta["tau"] + rest, axis=0)


def energy(image_size, batch, theta_energy):
    t = latency(image_size, batch, theta_energy)
    return (theta_energy["p_static"] * t
            + theta_energy["e_flop"] * flops(image_size, batch)
            + theta_energy["e_byte"] * bytes_moved(image_size, batch))

## Замеры

In [ ]:
BASE_S = [32, 64, 128, 224, 256, 384, 512]
BASE_B = [1, 2, 4, 8, 16, 32, 64, 128, 256]
SEED = 2026

COLUMNS = ["S", "B", "is_validation", "status", "latency_s", "latency_p10_s", "latency_p90_s",
           "n_latency", "memory_bytes", "mem_before_bytes", "energy_J", "avg_power_W", "n_energy",
           "flops_counted", "sm_clock_MHz", "temp_C"]


def make_grid(seed=SEED):
    rng = np.random.default_rng(seed)
    extra_s = sorted(rng.choice([s for s in range(32, 513, 16) if s not in BASE_S], 4, replace=False).tolist())
    extra_b = sorted(rng.choice([b for b in range(1, 257) if b & (b - 1)], 3, replace=False).tolist())
    grid = []
    for s in sorted(BASE_S + extra_s):
        for b in sorted(BASE_B + extra_b):
            grid.append((s, b, s in extra_s or b in extra_b))
    rng.shuffle(grid)
    return grid, extra_s, extra_b


def init_nvml():
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        pynvml.nvmlDeviceGetTotalEnergyConsumption(handle)
        return pynvml, handle
    except Exception as e:
        print("Счётчик энергии недоступен, энергию не меряем:", e)
        return None


def energy_joules(nvml):
    pynvml, handle = nvml
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(handle) / 1000


def idle_power(nvml, seconds=3.0):
    pynvml, handle = nvml
    values = []
    end = time.perf_counter() + seconds
    while time.perf_counter() < end:
        values.append(pynvml.nvmlDeviceGetPowerUsage(handle) / 1000)
        time.sleep(0.02)
    return float(np.median(values))


def run_forward(model, x):
    with torch.inference_mode():
        model(x)


def count_flops(model_meta, s, b):
    with FlopCounterMode(display=False) as counter, torch.inference_mode():
        model_meta(torch.empty(b, 3, s, s, device="meta"))
    return counter.get_total_flops()


def time_layers(model, x, reps):
    result = []
    with torch.inference_mode():
        for _ in range(reps):
            events = [torch.cuda.Event(enable_timing=True)]
            h = x
            torch.cuda.synchronize()
            events[0].record()
            for layer in model:
                h = layer(h)
                if not isinstance(layer, nn.Flatten):
                    events.append(torch.cuda.Event(enable_timing=True))
                    events[-1].record()
            torch.cuda.synchronize()
            result.append([events[i].elapsed_time(events[i + 1]) / 1000 for i in range(len(events) - 1)])
    return np.median(result, axis=0)


def measure_one(model, model_meta, nvml, s, b, device):
    row = {"S": s, "B": b, "flops_counted": count_flops(model_meta, s, b)}
    x = torch.randn(b, 3, s, s, device=device)

    start = time.perf_counter()
    n = 0
    while n < 3 or (time.perf_counter() - start < 0.3 and n < 50):
        run_forward(model, x)
        n += 1
    torch.cuda.synchronize()
    one_pass = (time.perf_counter() - start) / n

    torch.cuda.reset_peak_memory_stats()
    row["mem_before_bytes"] = torch.cuda.memory_allocated()
    run_forward(model, x)
    torch.cuda.synchronize()
    row["memory_bytes"] = torch.cuda.max_memory_allocated()

    times = []
    for _ in range(min(200, max(10, math.ceil(1.0 / one_pass)))):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        run_forward(model, x)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    row["latency_s"] = float(np.median(times))
    row["latency_p10_s"] = float(np.percentile(times, 10))
    row["latency_p90_s"] = float(np.percentile(times, 90))
    row["n_latency"] = len(times)

    if nvml:
        torch.cuda.synchronize()
        e0, t0, n = energy_joules(nvml), time.perf_counter(), 0
        while n < 3 or time.perf_counter() - t0 < 2.0:
            run_forward(model, x)
            n += 1
            if n % 8 == 0:
                torch.cuda.synchronize()
        torch.cuda.synchronize()
        joules, seconds = energy_joules(nvml) - e0, time.perf_counter() - t0
        pynvml, handle = nvml
        row["energy_J"] = joules / n
        row["avg_power_W"] = joules / seconds
        row["n_energy"] = n
        row["sm_clock_MHz"] = pynvml.nvmlDeviceGetClockInfo(handle, pynvml.NVML_CLOCK_SM)
        row["temp_C"] = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)

    layers = time_layers(model, x, 5 if row["latency_s"] < 0.1 else 3)
    return row, layers


def fits(model, s, b):
    x = None
    try:
        x = torch.randn(b, 3, s, s, device="cuda")
        run_forward(model, x)
        torch.cuda.synchronize()
        return True
    except torch.cuda.OutOfMemoryError:
        return False
    finally:
        del x
        gc.collect()
        torch.cuda.empty_cache()


def oom_probe(model, out):
    rows = []
    for s in (512, 384):
        gc.collect()
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        budget = free + torch.cuda.memory_allocated()
        predicted = int(max_batch(s, budget))
        lo = int(0.6 * predicted)
        hi = min(int(1.2 * predicted), (2**31 - 1) // (8 * s * s))
        lo_fits, hi_fits = fits(model, s, lo), fits(model, s, hi)
        measured = None
        if lo_fits and not hi_fits:
            while hi - lo > max(1, predicted // 200):
                mid = (lo + hi) // 2
                if fits(model, s, mid):
                    lo = mid
                else:
                    hi = mid
            measured = lo
        print(f"Граница OOM при S={s}: по формуле B_max = {predicted}, на деле {measured}")
        rows.append({"S": s, "budget_bytes": budget, "total_bytes": total, "B_max_predicted": predicted,
                     "B_max_measured": measured, "lo_fits": lo_fits, "hi_fits": hi_fits})
    with open(os.path.join(out, "oom_probe.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run_measurements(out="results", device="cuda", quick=False):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False

    os.makedirs(out, exist_ok=True)
    torch.manual_seed(0)
    model = make_model().to(device).eval()
    model_meta = make_model().to("meta").eval()
    nvml = init_nvml()

    grid, extra_s, extra_b = make_grid()
    if quick:
        grid = [g for g in grid if g[0] <= 64 and g[1] <= 8][:6]

    run_forward(model, torch.randn(2, 3, 64, 64, device=device))
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()

    env = {
        "gpu": torch.cuda.get_device_name(0),
        "gpu_total_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
        "driver": nvml[0].nvmlSystemGetDriverVersion() if nvml else None,
        "power_limit_W": nvml[0].nvmlDeviceGetPowerManagementLimit(nvml[1]) / 1000 if nvml else None,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "python": platform.python_version(),
        "seed": SEED,
        "random_S": extra_s,
        "random_B": extra_b,
        "idle_power_W": idle_power(nvml) if nvml else None,
        "allocated_after_warmup_bytes": torch.cuda.memory_allocated(),
    }
    with open(os.path.join(out, "env.json"), "w") as f:
        json.dump(env, f, indent=2)
    print(json.dumps(env, indent=2, ensure_ascii=False))

    meas_path = os.path.join(out, "measurements.csv")
    layers_path = os.path.join(out, "layer_times.csv")
    is_new = not os.path.exists(meas_path)
    done = set()
    if not is_new:
        with open(meas_path) as f:
            done = {(int(r["S"]), int(r["B"])) for r in csv.DictReader(f)}
    meas_file = open(meas_path, "a", newline="")
    layers_file = open(layers_path, "a", newline="")
    meas_writer = csv.DictWriter(meas_file, fieldnames=COLUMNS, restval="")
    layers_writer = csv.writer(layers_file)
    if is_new:
        meas_writer.writeheader()
        layers_writer.writerow(["S", "B", "layer_index", "layer", "time_s"])

    todo = [g for g in grid if (g[0], g[1]) not in done]
    start = time.perf_counter()
    for i, (s, b, is_val) in enumerate(todo, 1):
        gc.collect()
        torch.cuda.empty_cache()
        try:
            row, layers = measure_one(model, model_meta, nvml, s, b, device)
            row["status"] = "ok"
        except torch.cuda.OutOfMemoryError:
            row = {"S": s, "B": b, "status": "OOM", "memory_bytes": "OOM",
                   "flops_counted": count_flops(model_meta, s, b)}
            layers = None
        row["is_validation"] = int(is_val)
        meas_writer.writerow(row)
        meas_file.flush()
        if layers is not None:
            for j, t in enumerate(layers):
                layers_writer.writerow([s, b, j, LAYERS[j][0], f"{t:.9g}"])
            layers_file.flush()

        elapsed = time.perf_counter() - start
        if row["status"] == "OOM":
            print(f"[{i}/{len(todo)}] S={s} B={b}: не влезло в память ({elapsed:.0f} с)")
        else:
            print(f"[{i}/{len(todo)}] S={s} B={b}: {row['latency_s'] * 1e3:.3f} мс, "
                  f"{row['memory_bytes'] / 2**20:.1f} МиБ, {row.get('energy_J', float('nan')):.3g} Дж, "
                  f"{row.get('avg_power_W', float('nan')):.0f} Вт ({elapsed:.0f} с)", flush=True)
    meas_file.close()
    layers_file.close()

    if not quick:
        oom_probe(model, out)

In [ ]:
run_measurements()

## Калибровка

In [ ]:
def load(results):
    df = pd.read_csv(os.path.join(results, "measurements.csv"))
    df["oom"] = df["status"] == "OOM"
    for col in ["latency_s", "memory_bytes", "energy_J", "avg_power_W", "flops_counted"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values(["S", "B"]).reset_index(drop=True)

    layers = pd.read_csv(os.path.join(results, "layer_times.csv"))
    layers = layers.merge(df[["S", "B", "is_validation"]], on=["S", "B"])

    probe_path = os.path.join(results, "oom_probe.csv")
    probe = pd.read_csv(probe_path) if os.path.exists(probe_path) else None
    with open(os.path.join(results, "env.json")) as f:
        env = json.load(f)
    return df, layers, probe, env


def fit_bandwidth_throughput(layers):
    cal = layers[layers["is_validation"] == 0]
    f_all, m_all = layer_costs(cal["S"].values, cal["B"].values)
    idx = np.arange(len(cal))
    f = f_all[cal["layer_index"].values, idx]
    m = m_all[cal["layer_index"].values, idx]
    y = np.log(np.maximum(cal["time_s"].values, 1e-7))

    def residuals(p):
        floor, bw, perf = np.exp(p)
        return np.log(np.maximum(np.maximum(floor, m / bw), f / perf)) - y

    best = None
    for bw0 in [100e9, 250e9]:
        for perf0 in [2e12, 5e12]:
            res = least_squares(residuals, np.log([10e-6, bw0, perf0]), loss="soft_l1", f_scale=0.1)
            if best is None or res.cost < best.cost:
                best = res
    floor, bw, perf = np.exp(best.x)
    return float(floor), float(bw), float(perf)


def fit_overhead(df, bw, perf):
    cal = df[(df["is_validation"] == 0) & ~df["oom"]]
    s, b = cal["S"].values, cal["B"].values
    y = np.log(cal["latency_s"].values)

    def residuals(p):
        theta = {"t0": np.exp(p[0]), "tau": np.exp(p[1]), "bandwidth": bw, "throughput": perf}
        return np.log(latency(s, b, theta)) - y

    res = least_squares(residuals, np.log([30e-6, 15e-6]))
    return {"t0": float(np.exp(res.x[0])), "tau": float(np.exp(res.x[1])),
            "bandwidth": bw, "throughput": perf}


def fit_energy(df, theta):
    cal = df[(df["is_validation"] == 0) & ~df["oom"]].dropna(subset=["energy_J"])
    s, b, e = cal["S"].values, cal["B"].values, cal["energy_J"].values
    a = np.column_stack([latency(s, b, theta), flops(s, b), bytes_moved(s, b)])
    scale = a.max(axis=0)
    coef, _ = nnls(a / scale / e[:, None], np.ones(len(e)))
    coef = coef / scale
    return {**theta, "p_static": float(coef[0]), "e_flop": float(coef[1]), "e_byte": float(coef[2])}


def errors(df, theta, theta_e):
    ok = df[~df["oom"]]
    s, b = ok["S"].values, ok["B"].values
    val = ok["is_validation"].values == 1
    table = {}
    for name, pred, meas in [("latency", latency(s, b, theta), ok["latency_s"].values),
                             ("memory", memory(s, b), ok["memory_bytes"].values),
                             ("energy", energy(s, b, theta_e), ok["energy_J"].values),
                             ("flops", flops(s, b), ok["flops_counted"].values)]:
        err = np.abs(pred - meas) / meas * 100
        table[name] = {"calibration_mape": float(np.nanmean(err[~val])),
                       "validation_mape": float(np.nanmean(err[val])),
                       "validation_max": float(np.nanmax(err[val]))}
    return table


def run_calibration(results="results"):
    df, layers, probe, env = load(results)

    floor, bw, perf = fit_bandwidth_throughput(layers)
    theta = fit_overhead(df, bw, perf)
    theta_e = fit_energy(df, theta)

    budget = probe["budget_bytes"].iloc[0] if probe is not None else env["gpu_total_memory_bytes"]
    predicted_oom = memory(df["S"], df["B"]) > budget
    ok = df[~df["oom"]]

    out = {
        "latency": theta,
        "energy": theta_e,
        "layer_time_floor_s": floor,
        "errors_percent": errors(df, theta, theta_e),
        "oom": {"memory_budget_bytes": float(budget),
                "predicted": int(predicted_oom.sum()),
                "measured": int(df["oom"].sum()),
                "max_predicted_memory_bytes": float(memory(df["S"], df["B"]).max())},
        "corr_flops_bytes": float(np.corrcoef(flops(ok["S"], ok["B"]), bytes_moved(ok["S"], ok["B"]))[0, 1]),
    }
    with open(os.path.join(results, "theta.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(json.dumps(out, indent=2))

    make_all(df, layers, probe, env, theta, theta_e, os.path.join(results, "figures"))

## Графики

In [ ]:
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
T4 = {"throughput": 8.1e12, "bandwidth": 320e9}


def save(fig, folder, name):
    fig.tight_layout()
    fig.savefig(os.path.join(folder, name), dpi=150)
    plt.close(fig)


def points(ax, x, y, val, color):
    ax.scatter(x[~val], y[~val], color=color, s=16, zorder=3)
    ax.scatter(x[val], y[val], facecolor="white", edgecolor=color, marker="s", s=16, zorder=3)


def add_legend(ax):
    ax.scatter([], [], color="gray", s=16, label="замер, калибровка")
    ax.scatter([], [], facecolor="white", edgecolor="gray", marker="s", s=16, label="замер, валидация")
    ax.plot([], [], color="gray", label="формула")
    ax.legend(fontsize=7, loc="upper left")


def curves(fig, ax, df, column, predict, scale, by):
    groups = sorted(df[by].unique())
    norm = LogNorm(min(groups), max(groups))
    for g in groups:
        d = df[df[by] == g]
        color = plt.cm.viridis(0.9 * norm(g))
        if by == "S":
            x_line = np.geomspace(1, 256, 100)
            ax.plot(x_line, predict(g, x_line) * scale, color=color, lw=1)
            points(ax, d["B"].values, d[column].values * scale, d["is_validation"].values == 1, color)
        else:
            x_line = np.linspace(32, 512, 100)
            ax.plot(x_line, predict(x_line, g) * scale, color=color, lw=1)
            points(ax, d["S"].values, d[column].values * scale, d["is_validation"].values == 1, color)
    ax.set_xscale("log")
    ax.set_yscale("log")
    if by == "S":
        ax.set_xlabel("размер батча B")
        label = "размер картинки S"
    else:
        ax.set_xlabel("размер картинки S, пикс")
        ax.set_xticks([32, 64, 128, 256, 512], ["32", "64", "128", "256", "512"])
        label = "размер батча B"
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="viridis"), ax=ax, label=label)
    add_legend(ax)


def parity(ax, pred, meas, val, name, unit):
    points(ax, meas, pred, val, "tab:blue")
    lo, hi = min(meas.min(), pred.min()) / 1.5, max(meas.max(), pred.max()) * 1.5
    ax.plot([lo, hi], [lo, hi], "k-", lw=0.8)
    ax.plot([lo, hi], [lo * 0.8, hi * 0.8], "k:", lw=0.6)
    ax.plot([lo, hi], [lo * 1.25, hi * 1.25], "k:", lw=0.6)
    err = np.abs(pred - meas) / meas * 100
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(f"измерено, {unit}")
    ax.set_ylabel(f"по формуле, {unit}")
    ax.set_title(f"{name}: ошибка {np.nanmean(err[~val]):.1f}% (калибровка), "
                 f"{np.nanmean(err[val]):.1f}% (валидация)")
    ax.text(0.97, 0.03, "пунктир: ±25%", transform=ax.transAxes, ha="right", fontsize=7)


def plot_flops(df, folder):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    curves(fig, ax, df, "flops_counted", flops, 1.0, by="B")
    ax.set_ylabel("FLOPs за проход")
    err = np.abs(flops(df["S"], df["B"]) - df["flops_counted"]) / df["flops_counted"] * 100
    ax.set_title(f"FLOPs: формула и подсчёт PyTorch (макс. расхождение {err.max():.2g}%)")
    save(fig, folder, "flops.png")


def plot_memory(df, probe, budget, folder):
    ok = df[~df["oom"]]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    mib = 1 / 2**20

    curves(fig, axes[0], ok, "memory_bytes", memory, mib, by="B")
    axes[0].axhline(budget * mib, color="k", ls="--", lw=0.8)
    axes[0].text(512, budget * mib * 0.75, "доступная память GPU", fontsize=7, ha="right")
    oom = df[df["oom"]]
    if len(oom):
        axes[0].scatter(oom["S"], memory(oom["S"], oom["B"]) * mib, marker="x", color="red", s=30, zorder=4)
    axes[0].set_ylabel("пиковая память, МиБ")
    axes[0].set_title("Пиковая память")

    pred = memory(ok["S"], ok["B"])
    axes[1].scatter(pred * mib, ok["memory_bytes"] / pred, c=np.log2(ok["B"]), cmap="viridis", s=16)
    axes[1].axhline(1, color="k", lw=0.8)
    axes[1].set_xscale("log")
    axes[1].set_xlabel("по формуле, МиБ")
    axes[1].set_ylabel("измерено / по формуле")
    axes[1].set_title("Во сколько раз формула ошибается")

    if probe is not None:
        x = np.arange(len(probe))
        axes[2].bar(x - 0.2, max_batch(probe["S"], probe["budget_bytes"]), 0.4, label="по формуле", color="lightsteelblue")
        axes[2].bar(x + 0.2, probe["B_max_measured"], 0.4, label="на деле", color="tab:blue")
        axes[2].set_xticks(x, [f"S={s}" for s in probe["S"]])
        axes[2].set_ylabel("максимальный батч без OOM")
        axes[2].set_title("Граница OOM")
        axes[2].legend(fontsize=7)
    else:
        axes[2].axis("off")
    save(fig, folder, "memory.png")


def plot_three(df, column, predict, scale, name, unit, folder, filename):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    curves(fig, axes[0], df, column, predict, scale, by="S")
    curves(fig, axes[1], df, column, predict, scale, by="B")
    for ax in axes[:2]:
        ax.set_ylabel(f"{name}, {unit}")
    axes[0].set_title(f"{name} от B (линия на каждый S)")
    axes[1].set_title(f"{name} от S (линия на каждый B)")
    s, b = df["S"].values, df["B"].values
    parity(axes[2], predict(s, b) * scale, df[column].values * scale, df["is_validation"].values == 1, name, unit)
    save(fig, folder, filename)


def plot_surfaces(df, theta, theta_e, folder):
    ok = df[~df["oom"]]
    sg, bg = np.meshgrid(np.linspace(32, 512, 40), np.geomspace(1, 256, 40))
    items = [("задержка, мс", latency(sg, bg, theta) * 1e3, ok["latency_s"] * 1e3),
             ("память, МиБ", memory(sg, bg) / 2**20, ok["memory_bytes"] / 2**20),
             ("энергия, мДж", energy(sg, bg, theta_e) * 1e3, ok["energy_J"] * 1e3)]
    fig = plt.figure(figsize=(17, 5))
    for i, (name, z, meas) in enumerate(items, 1):
        ax = fig.add_subplot(1, 3, i, projection="3d")
        ax.plot_surface(np.log2(sg), np.log2(bg), np.log10(z), cmap="viridis", alpha=0.55, lw=0)
        m = meas.notna()
        ax.scatter(np.log2(ok["S"][m]), np.log2(ok["B"][m]), np.log10(meas[m]), color="k", s=6)
        ax.set_xlabel("log2 S")
        ax.set_ylabel("log2 B")
        ax.set_zlabel(f"log10 ({name})")
        ax.set_title(f"{name}: поверхность по формуле, точки из замеров")
    save(fig, folder, "surfaces.png")


def regime_shares(s, b, theta):
    f, m = layer_costs(s, b)
    t_mem = m / theta["bandwidth"]
    t_cmp = f / theta["throughput"]
    g = np.maximum(t_mem, t_cmp)
    rest = np.cumsum(g[::-1], axis=0)[::-1]
    k = np.arange(1, len(LAYERS) + 1)[:, None]
    first = np.argmax(k * theta["tau"] + rest, axis=0)
    on_gpu = np.arange(len(LAYERS))[:, None] >= first
    launch = theta["t0"] + (first + 1) * theta["tau"]
    mem = (g * on_gpu * (t_mem >= t_cmp)).sum(axis=0)
    comp = (g * on_gpu * (t_mem < t_cmp)).sum(axis=0)
    total = launch + mem + comp
    return launch / total, mem / total, comp / total


def plot_regimes(df, theta, folder):
    ok = df[~df["oom"]].copy()
    ok["x"] = ok["B"] * ok["S"] ** 2
    ok = ok.sort_values("x")
    launch, mem, comp = regime_shares(ok["S"].values, ok["B"].values, theta)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].stackplot(ok["x"], launch, mem, comp, colors=["silver", "tab:blue", "tab:red"],
                      labels=["запуск ядер (CPU)", "упор в память", "упор в вычисления"])
    axes[0].set_xscale("log")
    axes[0].set_ylim(0, 1)
    axes[0].set_xlabel("B·S², пикселей в батче")
    axes[0].set_ylabel("доля времени по формуле")
    axes[0].set_title("На что уходит время")
    axes[0].legend(fontsize=7, loc="center left")

    points(axes[1], ok["x"].values, ok["latency_s"].values / ok["x"].values * 1e9,
           ok["is_validation"].values == 1, "tab:blue")
    for s in [32, 128, 512]:
        b = np.geomspace(1, 256, 100)
        axes[1].plot(b * s**2, latency(s, b, theta) / (b * s**2) * 1e9, lw=1, label=f"формула, S={s}")
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("B·S², пикселей в батче")
    axes[1].set_ylabel("время на один пиксель входа, нс")
    axes[1].set_title("Время на пиксель")
    axes[1].legend(fontsize=7)
    save(fig, folder, "regimes.png")


def plot_layers(layers, theta, gpu, folder):
    f_all, m_all = layer_costs(layers["S"].values, layers["B"].values)
    idx = np.arange(len(layers))
    layers = layers.assign(F=f_all[layers["layer_index"].values, idx], M=m_all[layers["layer_index"].values, idx])
    layers["kind"] = [LAYERS[i][1] for i in layers["layer_index"]]
    busy = layers[layers["time_s"] > 5 * layers.groupby("layer_index")["time_s"].transform("min")]

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    compute = busy[busy["kind"] != "mem"]
    for i in sorted(compute["layer_index"].unique()):
        d = compute[compute["layer_index"] == i]
        axes[0].scatter(d["F"] / d["M"], d["F"] / d["time_s"], s=10, label=LAYERS[i][0])
    ai = np.geomspace(1, 1000, 100)
    axes[0].plot(ai, np.minimum(theta["throughput"], theta["bandwidth"] * ai), "k-", lw=1, label="подобранный roofline")
    if "T4" in gpu:
        axes[0].plot(ai, np.minimum(T4["throughput"], T4["bandwidth"] * ai), "k--", lw=0.8, label="паспорт T4")
    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("арифметическая интенсивность, FLOP/байт")
    axes[0].set_ylabel("достигнуто, FLOP/с")
    axes[0].set_title("Roofline: свёртки и Linear")
    axes[0].legend(fontsize=7)

    mem = busy[busy["kind"] == "mem"]
    for i in sorted(mem["layer_index"].unique()):
        d = mem[mem["layer_index"] == i]
        axes[1].scatter(d["M"], d["M"] / d["time_s"] / 1e9, s=10, label=LAYERS[i][0])
    axes[1].axhline(theta["bandwidth"] / 1e9, color="k", lw=1, label="подобранная β")
    if "T4" in gpu:
        axes[1].axhline(T4["bandwidth"] / 1e9, color="k", ls="--", lw=0.8, label="паспорт T4")
    axes[1].set_xscale("log")
    axes[1].set_xlabel("байт перекладывает слой")
    axes[1].set_ylabel("достигнутая скорость памяти, ГБ/с")
    axes[1].set_title("ReLU, pooling, GAP")
    axes[1].legend(fontsize=6, ncol=2)

    s_max = layers["S"].max()
    b_max = layers[layers["S"] == s_max]["B"].max()
    d = layers[(layers["S"] == s_max) & (layers["B"] == b_max)].sort_values("layer_index")
    pred = np.maximum(d["M"] / theta["bandwidth"], d["F"] / theta["throughput"])
    y = np.arange(len(d))
    axes[2].barh(y - 0.2, d["time_s"] * 1e3, 0.4, label="замер", color="tab:blue")
    axes[2].barh(y + 0.2, pred * 1e3, 0.4, label="формула", color="lightsteelblue")
    axes[2].set_yticks(y, [LAYERS[i][0] for i in d["layer_index"]])
    axes[2].invert_yaxis()
    axes[2].set_xlabel("время, мс")
    axes[2].set_title(f"Время по слоям, S={s_max}, B={b_max}")
    axes[2].legend(fontsize=7)
    save(fig, folder, "layers.png")


def make_all(df, layers, probe, env, theta, theta_e, folder):
    os.makedirs(folder, exist_ok=True)
    ok = df[~df["oom"]]
    budget = probe["budget_bytes"].iloc[0] if probe is not None else env["gpu_total_memory_bytes"]
    plot_flops(df, folder)
    plot_memory(df, probe, budget, folder)
    plot_three(ok, "latency_s", lambda s, b: latency(s, b, theta), 1e3, "задержка", "мс", folder, "latency.png")
    plot_three(ok.dropna(subset=["energy_J"]), "energy_J", lambda s, b: energy(s, b, theta_e), 1e3,
               "энергия", "мДж", folder, "energy.png")
    plot_surfaces(df, theta, theta_e, folder)
    plot_regimes(df, theta, folder)
    plot_layers(layers, theta, env.get("gpu", ""), folder)
    print("Графики сохранены в", folder)

In [ ]:
run_calibration()
for p in sorted(glob.glob("results/figures/*.png")):
    print(p)
    display(Image(p))

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')